In [ ]:
from pathlib import Path

import pandas as pd
import missingno as msno
import numpy as np

data_dir = Path("../data")

In [ ]:
df = pd.read_csv(data_dir / "raw/AllClinical00.csv")
df2 = pd.read_csv(data_dir / "processed/AllClinical00_variable_inventory.csv")

In [ ]:
# Which columns have actual missing values
df.isna().sum().loc[lambda x: x > 0]


In [ ]:
blank   = df.isna()
coded   = df.astype(str).apply(lambda s: s.str.startswith(".:"))

# Cells with either type of missing
missing = blank | coded

summary = pd.DataFrame({
    "blank" : blank.sum(),
    "coded" : coded.sum(),
})

# 
summary = summary[summary.sum(axis=1) > 0]

summary["dataset"] = summary.index.map(
    df2.set_index("AllClinical00")["Dataset"]
)

print(summary.head())

In [ ]:
summary["total_missing"] = (summary["blank"] + summary["coded"])

summary["missing_pct"] = summary["total_missing"] / len(df) * 100


summary.sort_values("missing_pct", ascending=False).head(50)

In [ ]:
dataset_missing = (
    summary.groupby("dataset").agg(
        columns=("dataset", "size"),
        total_missing=("total_missing", "sum"),
        avg_missing_pct=("missing_pct", "mean"),
    )
    .sort_values("avg_missing_pct", ascending=False)
)
dataset_missing

In [ ]:
n = (summary["missing_pct"] > 80).sum()
print(f"{n} columns have more than 80% missing values")

In [ ]:
#summary[summary["missing_pct"] > 80]